# Footprint decomposition — multi-scenario synthesis

Sectoral breakdown of the environmental footprints (planetary boundaries,
LP) by consumption indicator, for the base scenario and the 2050
transition scenarios.

Generates 2 types of figures in `figures/footprint_decomposition/`, each one
with the Excel workbook of its data (same name, `.xlsx`):
- 1 stacked bar in % per scenario (`scenario_NN_<name>.png`, `create_footprint_decomposition_by_scenario`)
- 1 synthesis figure for all the scenarios, absolute values with LP/DLS boundaries (`synthesis_by_lp_all_scenarios.png`, `create_footprint_decomposition`)

Computation and plotting logic shared with the other notebooks through the
[`planefr_lib`](./planefr_lib/) package.

In [ ]:
# ============================ PARAMETERS ============================
SCENARIO_GROUP = "CONF"           # France scenarios (e.g. "CONF", "ADEME", "no_SSP", "SSP2_2050"...)
SHARING_PRINCIPLE = "EPC"         # national budget = world budget x share in budget_shares.xlsx:
                                  #   "EPC" (equal per capita) or "CTR" (capability to reduce)
ADD_2019_FR = True                # (fig. 3) True: also one figure for 2019_FR (2019_raw/2019_FR)
GROUP_BY_CATEGORY_SCENARIO = True    # (fig. 3) stacking: True = cat1-dom, cat1-imp, cat2-dom...; False = all dom then all imp
GROUP_BY_CATEGORY_SYNTHESIS = False  # (fig. 4) same choice for the synthesis figure
SHOW_VALUES = True                # True: values / overshoot levels annotated on the bars
SHOW_PBA = True                   # (fig. 4) True: production-based accounting (PBA) cross shown
SHOW_DLS = True                   # (fig. 4) True: Decent Living Standards (social floor) shown
DISPLAY_BOUNDS = True             # (fig. 4) True: Lower/High-Risk Boundary shown in addition to the Safe Boundary
SHOW_VARIATION_BARS = True        # (fig. 4) True: non-reference scenarios drawn as variations vs the reference
DPI = 300                         # resolution of the saved .png
# ====================================================================

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

notebooks_dir = Path.cwd()
while not (notebooks_dir / "planefr_lib").exists() and notebooks_dir != notebooks_dir.parent:
    notebooks_dir = notebooks_dir.parent
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

import matplotlib.pyplot as plt

from planefr_lib import config, io, processing
from planefr_lib.plot_stacked_bar import create_footprint_decomposition_by_scenario, create_footprint_decomposition

# Folder of the figures (and of their Excel workbooks): figures/<NOTEBOOK_NAME>/
NOTEBOOK_NAME = "footprint_decomposition"
figures_dir = config.figure_dir(NOTEBOOK_NAME)

## 1. Loading the common data

In [ ]:
bridge_matrices_df = io.load_bridge_matrices()
dls_df = io.load_dls()
cf_df = io.load_characterization_factors(SCENARIO_GROUP)
boundaries_df = io.load_boundaries(SCENARIO_GROUP)
scenario_folders = config.get_scenario_folders(exclude_2019=True, scenario_group=SCENARIO_GROUP)
scenario_names = [f.name for f in scenario_folders]
print(f"Scenarios found ({len(scenario_folders)}): {scenario_names}")

## 2. Processing of all the scenarios

In [ ]:
all_scenarios_data = []
all_indicator_to_lp = []
for scenario_folder in scenario_folders:
    print(f"Processing: {scenario_folder.name}")
    data_by_indicator, indicator_to_lp = processing.process_scenario(
        scenario_folder, cf_df, bridge_matrices_df, boundaries_df)
    all_scenarios_data.append(data_by_indicator)
    all_indicator_to_lp.append(indicator_to_lp)
assert len(all_scenarios_data) == len(scenario_folders), "Some scenarios produced no data"
print(f"\n{len(all_scenarios_data)}/{len(scenario_folders)} scenarios processed successfully.")

## 3. Per-scenario figure (stacked bars, %)

In [ ]:
# 2019_FR is only added to this per-scenario figure (not to the synthesis of step 4).
items = list(zip(scenario_folders, all_scenarios_data, all_indicator_to_lp))
if ADD_2019_FR:
    folder_2019 = config.RAW_2019_DIR / "2019_FR"
    items.append((folder_2019, *processing.process_scenario(folder_2019, cf_df, bridge_matrices_df, boundaries_df)))

for scenario_idx, (scenario_folder, data_by_indicator, indicator_to_lp) in enumerate(items):
    output_path = figures_dir / f"scenario_{scenario_idx:02d}_{scenario_folder.name}.png"
    fig, ax = create_footprint_decomposition_by_scenario(
        data_by_indicator, boundaries_df, indicator_to_lp,
        scenario_name=scenario_folder.name, group_by_category=GROUP_BY_CATEGORY_SCENARIO,
        sharing_principle=SHARING_PRINCIPLE, show_values=SHOW_VALUES,
        excel_path=output_path.with_suffix(".xlsx"))
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.show()
    print(f"Saved: {output_path.name}")

## 4. Synthesis figure

In [ ]:
output_path = figures_dir / "synthesis_by_lp_all_scenarios.png"
fig_by_lp, _ = create_footprint_decomposition(
    all_scenarios_data, boundaries_df, all_indicator_to_lp, scenario_names, dls_df,
    group_by_category=GROUP_BY_CATEGORY_SYNTHESIS, show_pba=SHOW_PBA, show_dls=SHOW_DLS,
    show_values=SHOW_VALUES, display_bounds=DISPLAY_BOUNDS,
    show_variation_bars=SHOW_VARIATION_BARS, sharing_principle=SHARING_PRINCIPLE,
    scenario_group=SCENARIO_GROUP, excel_path=output_path.with_suffix(".xlsx"))
fig_by_lp.savefig(output_path, dpi=DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path.name}")